NOTE: A similar validation is performed in `test_model.ipynb`

In [2]:
from pathlib import Path
import pandas as pd
from typing import Literal
import copy

from phd_visualizations.regression import regression_plot
from phd_visualizations.calculations import SupportedInstruments 
from phd_visualizations import save_figure

import combined_cooler
import matlab

%reload_ext autoreload
%autoreload 2

data_path = Path("../data")
model_type: Literal["physical", "data"] = "physical"

df = pd.read_csv(data_path / "cc_out_exp.csv")
df["Tcc_out"] = df["Tc_in"]
df


,date,Tv,Tamb,HR,qc,Tc_in,Ce_cc,Ce,Tcond,Tdc_in,...,mv,Tdc_out,Twct_out,wdc,wwct,qdc,qwct,Rp,Rs,Tcc_out
0,19-Feb-2025,36.261009,17.671711,43.180898,23.998493,28.121629,0.450381,3.653911,35.805044,33.494083,...,224.076934,28.047818,27.778468,31.974352,22.470002,12.417720,12.259674,0.483,0.054,28.121629
1,10-Feb-2025,37.774713,13.531843,61.437262,24.000376,27.700450,0.432014,3.442630,37.052174,34.112194,...,276.536959,27.430885,27.571514,30.637523,23.770823,12.005001,11.779023,0.500,0.000,27.700450
2,12-Feb-2025,36.241372,19.685473,23.739778,23.999315,28.145370,0.381511,4.625762,35.705996,21.977082,...,217.364415,20.147901,27.961559,0.000000,39.757368,0.235675,23.694971,0.990,0.000,28.145370
3,19-Feb-2025,37.575024,19.473600,38.711528,23.998248,27.738758,0.667380,4.861102,36.284572,31.971167,...,263.510629,23.391675,27.584572,0.000000,50.849622,0.390683,23.691506,0.984,0.201,27.738758
4,08-Jul-2025,37.389592,27.213752,50.149914,23.999880,28.115154,7.107880,10.170240,36.956869,34.384180,...,258.600484,30.136470,24.924979,100.000000,66.439999,13.998713,9.773220,0.417,0.000,28.115154
5,08-Jul-2025,38.576245,28.395690,42.827841,24.001603,28.341443,7.863237,10.932670,38.064432,35.242784,...,270.261573,31.153257,24.004598,100.000000,81.709604,14.000688,9.770016,0.417,0.000,28.341443
6,09-Jul-2025,42.399588,27.275200,62.124991,24.000049,35.196531,0.383408,4.809472,42.201587,25.526867,...,200.822004,25.948971,35.006526,0.000000,39.846699,0.119232,23.694192,0.995,0.000,35.196531
7,09-Jul-2025,39.453322,31.500935,41.542744,23.999335,28.792534,8.163791,11.842760,38.904821,35.742446,...,280.806774,30.781717,27.996002,100.000000,86.901059,5.695932,18.024022,0.763,0.000,28.792534
8,13-Jan-2025,44.360271,12.164853,48.714282,24.002221,37.228958,0.271859,3.893380,43.848984,42.061851,...,214.037281,36.973783,15.001290,30.000000,0.000000,23.798313,0.000540,0.008,0.000,37.228958
9,02-Dec-2024,43.324459,20.923333,42.056077,17.998552,30.498067,1.947239,3.261430,42.871410,40.086938,...,303.464304,30.478507,29.973642,59.506664,24.675237,11.778898,6.036892,0.346,0.000,30.498067


### Evaluate model

In [3]:
cc_model = combined_cooler.initialize()

print(f"Using MATLAB package: {cc_model.name}")

parameters_default = cc_model.default_parameters(nargout=1)

results_dfs = []
for model_type in ["data", "physical"]:
    params = copy.deepcopy(parameters_default)
    if model_type == "physical":
        
        params["dc_lb"] = matlab.double([[5.0600,   10.0, 5.2211, 11]])
        params["dc_ub"] = matlab.double([[50.7500,   50.0, 24.1543, 99.1800]])

        # WCT           "Tamb",     "HR",    "Tin",      "q",     "w_fan"
        params["wct_lb"] = matlab.double([[0.1,    0.1,     5.0,    5.0,       0.]])
        params["wct_ub"] = matlab.double([[50.0,   99.99,   55.0,   24.8400,  95.]])
    elif model_type == "data":    
        params["dc_model_data_path"] = "/workspaces/SOLhycool/modeling/matlab/component_models/dc_model_data.mat"
        params["wct_model_data_path"] = "/workspaces/SOLhycool/modeling/matlab/component_models/wct_model_data.mat"
    results: list[dict] = []
    for idx, ds in df.iterrows():

        qc_m3h = matlab.double([ds["qc"]])
        qdc_m3h = matlab.double([ds["qdc"]])
        qwct_m3h = matlab.double([ds["qwct"]])

        Rp, Rs = cc_model.flows_to_ratios(qc_m3h, qdc_m3h, qwct_m3h, nargout=2)

        Tamb_C = matlab.double([ds["Tamb"]])
        HR_pp = matlab.double([ds["HR"]])
        mv_kgh = matlab.double([ds["mv"]])
        wdc = matlab.double([ds["wdc"]])
        wwct = matlab.double([ds["wwct"]])
        Tv = matlab.double([ds["Tv"]])

        # Create the 'options' struct as a Python dictionary
        options = {
            'model_type': model_type,
            'parameters': params,  # Empty struct in MATLAB
            'lb': [], 'ub': [], 'x0': [], 'silence_warnings': True
        }

        Ce_kWe, Cw_lh, detailed = cc_model.combined_cooler_model(Tamb_C, HR_pp, mv_kgh, qc_m3h, Rp, Rs, wdc, wwct, Tv, options, nargout=3)
        results.append(detailed)
        
    results_dfs.append( pd.DataFrame(results) )

display(results_dfs[0])
display(results_dfs[1])


Authorization required, but no authorization protocol specified



Using MATLAB package: combined_cooler


,Tamb,HR,mv,qc,Rp,Rs,wdc,wwct,Ce,Cw,...,Tdc_in,Tdc_out,Ce_dc,Qdc,qwct,Twct_in,Twct_out,Ce_wct,Cw_wct,Qwct
0,17.671711,43.180898,224.076934,23.998493,0.483,0.054,31.974352,22.470002,455.027181,92.594846,...,33.159526,27.835118,312.366224,76.697196,12.261262,32.868585,27.490733,138.014403,92.594846,76.557338
1,13.531843,61.437262,276.536959,24.000376,0.500,0.000,30.637523,23.770823,436.661507,112.937130,...,33.821712,27.738266,284.240900,84.754647,12.000188,33.821712,26.948431,147.773128,112.937130,95.761066
2,19.685473,23.739778,217.364415,23.999315,0.990,0.000,0.000000,39.757368,386.157899,166.802855,...,33.278941,33.278941,0.000000,0.000000,23.759322,33.278941,28.959686,381.510941,166.802855,119.140562
3,19.473600,38.711528,263.510629,23.998248,0.984,0.201,0.000000,50.849622,672.026501,214.568197,...,33.888814,33.888814,0.000000,0.000000,23.691455,33.888814,28.832295,667.380066,214.568197,139.076575
4,27.213752,50.149914,258.600484,23.999880,0.417,0.000,100.000000,66.439999,7112.527479,133.927631,...,33.771250,29.372727,5867.400000,71.447959,10.007950,33.771250,23.766667,1240.480243,133.927631,116.260286
5,28.395690,42.827841,270.261573,24.001603,0.417,0.000,100.000000,81.709604,7867.885245,198.479095,...,34.989055,30.545062,5867.400000,72.187182,10.008669,34.989055,24.098859,1995.837164,198.479095,126.553453
6,27.275200,62.124991,200.822004,24.000049,0.995,0.000,0.000000,39.846699,388.055750,214.035596,...,40.385228,40.385228,0.000000,0.000000,23.880049,40.385228,34.774157,383.408432,214.035596,155.521376
7,31.500935,41.542744,280.806774,23.999335,0.763,0.000,100.000000,86.901059,8168.437540,331.331028,...,35.843477,31.074363,5867.400000,31.488302,18.311492,35.843477,28.501615,2296.390573,331.331028,156.070547
8,12.164853,48.714282,214.037281,24.002221,0.008,0.000,30.000000,0.000000,276.507384,0.000000,...,42.299130,37.589103,271.859000,130.162140,0.192018,42.299130,42.299130,0.000000,0.000000,0.000000
9,20.923333,42.056077,303.464304,17.998552,0.346,0.000,59.506664,24.675237,1949.495588,116.927160,...,39.959556,30.477610,1791.859404,129.553037,6.227499,39.959556,29.040120,155.379328,116.927160,78.933167


,Tamb,HR,mv,qc,Rp,Rs,wdc,wwct,Ce,Cw,...,Tdc_in,Tdc_out,Ce_dc,Qdc,qwct,Twct_in,Twct_out,Ce_wct,Cw_wct,Qwct
0,17.671711,43.180898,224.076934,23.998493,0.483,0.054,31.974352,22.470002,455.027181,95.530043,...,33.159526,28.069098,312.366224,73.326200,12.261262,32.881371,27.294913,138.014403,95.530043,79.527473
1,13.531843,61.437262,276.536959,24.000376,0.500,0.000,30.637523,23.770823,436.661507,111.887307,...,33.821712,27.215875,284.240900,92.034140,12.000188,33.821712,26.969546,147.773128,111.887307,95.466819
2,19.685473,23.739778,217.364415,23.999315,0.990,0.000,0.000000,39.757368,386.157899,195.773397,...,33.278941,33.278941,0.000000,0.000000,23.759322,33.278941,27.600132,381.510941,195.773397,156.648623
3,19.473600,38.711528,263.510629,23.998248,0.984,0.201,0.000000,50.849622,672.026501,225.389836,...,33.888814,33.888814,0.000000,0.000000,23.691455,33.888814,27.224474,667.380066,225.389836,183.307771
4,27.213752,50.149914,258.600484,23.999880,0.417,0.000,100.000000,66.439999,7112.527479,149.206450,...,33.771250,29.688823,5867.400000,66.312809,10.007950,33.771250,24.469662,1240.480243,149.206450,108.088079
5,28.395690,42.827841,270.261573,24.001603,0.417,0.000,100.000000,81.709604,7867.885245,182.263510,...,34.989055,30.877860,5867.400000,66.780750,10.008669,34.989055,23.918692,1995.837164,182.263510,128.647991
6,27.275200,62.124991,200.822004,24.000049,0.995,0.000,0.000000,39.846699,388.055750,219.390853,...,40.385228,40.385228,0.000000,0.000000,23.880049,40.385228,34.241805,383.408432,219.390853,170.277363
7,31.500935,41.542744,280.806774,23.999335,0.763,0.000,100.000000,86.901059,8168.437540,241.868112,...,35.843477,32.158253,5867.400000,24.331311,18.311492,35.843477,27.914413,2296.390573,241.868112,168.555730
8,12.164853,48.714282,214.037281,24.002221,0.008,0.000,30.000000,0.000000,276.507384,0.000000,...,42.299130,36.823954,271.859000,151.307410,0.192018,42.299130,42.299130,0.000000,0.000000,0.000000
9,20.923333,42.056077,303.464304,17.998552,0.346,0.000,59.506664,24.675237,1949.495588,119.041564,...,39.959556,29.958103,1791.859404,136.652312,6.227499,39.959556,26.671981,155.379328,119.041564,96.056359


In [4]:
# Export resutlts
for model_type, results_df in zip(["data", "physical"], results_dfs):
    results_df.to_csv(data_path / f"cc_out_mod_{model_type}.csv", index=False)
    print(f"Exported {data_path / f'cc_out_mod_{model_type}.csv'}")


Exported ../data/cc_out_mod_data.csv
Exported ../data/cc_out_mod_physical.csv


### Take from pre-evaluated

In [41]:
df = pd.read_csv(data_path / "cc_out_exp.csv")
results_df = pd.read_csv(data_path / "cc_out_mod_physical.csv")

df["Qdc"] = results_df["Qdc"]
df["Qwct"] = results_df["Qwct"]
df["Qc"] = results_df["Qc_released"]

display(df)
display(results_df)


,date,Tv,Tamb,HR,qc,Tc_in,Ce_cc,Ce,Tcond,Tdc_in,...,Twct_out,wdc,wwct,qdc,qwct,Rp,Rs,Qdc,Qwct,Qc
0,19-Feb-2025,36.261009,17.671711,43.180898,23.998493,28.121629,0.450381,3.653911,35.805044,33.494083,...,27.778468,31.974352,22.470002,12.417720,12.259674,0.483,0.054,73.326200,79.527473,150.314103
1,10-Feb-2025,37.774713,13.531843,61.437262,24.000376,27.700450,0.432014,3.442630,37.052174,34.112194,...,27.571514,30.637523,23.770823,12.005001,11.779023,0.500,0.000,92.034140,95.466819,185.227531
2,12-Feb-2025,36.241372,19.685473,23.739778,23.999315,28.145370,0.381511,4.625762,35.705996,21.977082,...,27.961559,0.000000,39.757368,0.235675,23.694971,0.990,0.000,0.000000,156.648623,145.814074
3,19-Feb-2025,37.575024,19.473600,38.711528,23.998248,27.738758,0.667380,4.861102,36.284572,31.971167,...,27.584572,0.000000,50.849622,0.390683,23.691506,0.984,0.201,0.000000,183.307771,176.537254
4,08-Jul-2025,37.389592,27.213752,50.149914,23.999880,28.115154,7.107880,10.170240,36.956869,34.384180,...,24.924979,100.000000,66.439999,13.998713,9.773220,0.417,0.000,66.312809,108.088079,173.279537
5,08-Jul-2025,38.576245,28.395690,42.827841,24.001603,28.341443,7.863237,10.932670,38.064432,35.242784,...,24.004598,100.000000,81.709604,14.000688,9.770016,0.417,0.000,66.780750,128.647991,180.880465
6,09-Jul-2025,42.399588,27.275200,62.124991,24.000049,35.196531,0.383408,4.809472,42.201587,25.526867,...,35.006526,0.000000,39.846699,0.119232,23.694192,0.995,0.000,0.000000,170.277363,133.895591
7,09-Jul-2025,39.453322,31.500935,41.542744,23.999335,28.792534,8.163791,11.842760,38.904821,35.742446,...,27.996002,100.000000,86.901059,5.695932,18.024022,0.763,0.000,24.331311,168.555730,187.774619
8,13-Jan-2025,44.360271,12.164853,48.714282,24.002221,37.228958,0.271859,3.893380,43.848984,42.061851,...,15.001290,30.000000,0.000000,23.798313,0.000540,0.008,0.000,151.307410,0.000000,142.427062
9,02-Dec-2024,43.324459,20.923333,42.056077,17.998552,30.498067,1.947239,3.261430,42.871410,40.086938,...,29.973642,59.506664,24.675237,11.778898,6.036892,0.346,0.000,136.652312,96.056359,202.144133


,Tamb,HR,mv,qc,Rp,Rs,wdc,wwct,Ce,Cw,...,Tdc_in,Tdc_out,Ce_dc,Qdc,qwct,Twct_in,Twct_out,Ce_wct,Cw_wct,Qwct
0,17.671711,43.180898,224.076934,23.998493,0.483,0.054,31.974352,22.470002,455.027181,95.530043,...,33.159526,28.069098,312.366224,73.326200,12.261262,32.881371,27.294913,138.014403,95.530043,79.527473
1,13.531843,61.437262,276.536959,24.000376,0.500,0.000,30.637523,23.770823,436.661507,111.887307,...,33.821712,27.215875,284.240900,92.034140,12.000188,33.821712,26.969546,147.773128,111.887307,95.466819
2,19.685473,23.739778,217.364415,23.999315,0.990,0.000,0.000000,39.757368,386.157899,195.773397,...,33.278941,33.278941,0.000000,0.000000,23.759322,33.278941,27.600132,381.510941,195.773397,156.648623
3,19.473600,38.711528,263.510629,23.998248,0.984,0.201,0.000000,50.849622,672.026501,225.389836,...,33.888814,33.888814,0.000000,0.000000,23.691455,33.888814,27.224474,667.380066,225.389836,183.307771
4,27.213752,50.149914,258.600484,23.999880,0.417,0.000,100.000000,66.439999,7112.527479,149.206450,...,33.771250,29.688823,5867.400000,66.312809,10.007950,33.771250,24.469662,1240.480243,149.206450,108.088079
5,28.395690,42.827841,270.261573,24.001603,0.417,0.000,100.000000,81.709604,7867.885245,182.263510,...,34.989055,30.877860,5867.400000,66.780750,10.008669,34.989055,23.918692,1995.837164,182.263510,128.647991
6,27.275200,62.124991,200.822004,24.000049,0.995,0.000,0.000000,39.846699,388.055750,219.390853,...,40.385228,40.385228,0.000000,0.000000,23.880049,40.385228,34.241805,383.408432,219.390853,170.277363
7,31.500935,41.542744,280.806774,23.999335,0.763,0.000,100.000000,86.901059,8168.437540,241.868112,...,35.843477,32.158253,5867.400000,24.331311,18.311492,35.843477,27.914413,2296.390573,241.868112,168.555730
8,12.164853,48.714282,214.037281,24.002221,0.008,0.000,30.000000,0.000000,276.507384,0.000000,...,42.299130,36.823954,271.859000,151.307410,0.192018,42.299130,42.299130,0.000000,0.000000,0.000000
9,20.923333,42.056077,303.464304,17.998552,0.346,0.000,59.506664,24.675237,1949.495588,119.041564,...,39.959556,29.958103,1791.859404,136.652312,6.227499,39.959556,26.671981,155.379328,119.041564,96.056359


In [47]:
from phd_visualizations.super_makers import SuperMarker

fig = regression_plot(
    df_ref= df, 
    df_mod= results_dfs[-1], 
    alternative_labels="Physical", # "Data", 
    var_ids=["Tdc_out", "Twct_out", "Tc_in", "Tc_out", "Cw"], 
    units=["℃"]*4 + ["l/h"], 
    instruments=[SupportedInstruments.pt100]*4 + [SupportedInstruments.paddle_wheel_flow_meter],
    show_error_metrics=["r2", "mae"],
    var_labels=[
        '<span style="color:#83b366; font-weight:bold;">DC</span> outlet temperature', 
        '<span style="color:#9573a6; font-weight:bold;">WCT</span> outlet temperature', 
        "Condenser inlet temp.", 
        "Condenser outlet temp.", 
        "Water consumption"
    ],
    legend_pos="top_spaced",
    inline_error_metrics_text=True,
    
    # kwargs for layout
    title_text='<span style="color:#83b366; font-weight:bold;">Combined</span> <span style="color:#9573a6; font-weight:bold;">cooling</span> model validation',
    title_font_size=21,
    legend_y = 1.015,
    title_y=0.995,
    # legend_font=dict(size=12),
    width=445,
    height=1850,
    vertical_spacing=0.06,
    margin=dict(l=10, r=10, t=120, b=10),
    
    super_marker=SuperMarker(
        size_var_id="Qc",
        size_var_range=[df["Qc"].min(), df["Qc"].max()],
        size_label="Cooling power",
        
        fill_var_id="Tamb",
        fill_label="Ambient temperature",
        fill_var_range=[10, 35],
        
        ring_vars_ids=["Qwct", "Qdc"] # TODO: Add Qdc, Qwct
    )
)

fig


In [48]:
save_figure(
    fig,
    figure_name="cc_model_regression",
    figure_path=Path("../results"),
    formats=["png", "svg"],
)


2025-07-17 18:32:07.707 | INFO     | phd_visualizations:save_figure:41 - Figure saved in ../results/cc_model_regression.png
2025-07-17 18:32:08.413 | INFO     | phd_visualizations:save_figure:41 - Figure saved in ../results/cc_model_regression.svg
